# 🛰️ Siamese-SNN Change Detection — V3 (FINAL)

**Key Fix**: Test cities had NO labels (cm.png missing) → F1 was always 0.
Now uses proper train/val split from labeled cities only.

**Runtime → Change runtime type → T4 GPU**

In [ ]:
!pip install -q torch torchvision snntorch rasterio pillow scipy tqdm

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ No GPU! Runtime → Change runtime type → T4 GPU')

In [ ]:
from google.colab import files
import os
from pathlib import Path

print('Upload your oscd_dataset.zip:')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
!mkdir -p /content/data/oscd
!unzip -q -o "{zip_name}" -d /content/data/oscd/

oscd_root = Path('/content/data/oscd')
for c in ['aguasclaras','bercy','paris','mumbai']:
    if (oscd_root/c).exists(): break
    for sub in oscd_root.iterdir():
        if sub.is_dir() and (sub/c).exists():
            oscd_root = sub; break

# Verify labels
labeled = [d.name for d in oscd_root.iterdir() if d.is_dir() and (d/'cm'/'cm.png').exists()]
unlabeled = [d.name for d in oscd_root.iterdir() if d.is_dir() and not (d/'cm'/'cm.png').exists() and d.name not in ('train','test')]
print(f'\n✅ {len(labeled)} cities WITH labels: {sorted(labeled)}')
print(f'❌ {len(unlabeled)} cities WITHOUT labels: {sorted(unlabeled)}')
print(f'\nDataset root: {oscd_root}')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Tuple, Optional
import numpy as np

try:
    import snntorch as snn
    from snntorch import surrogate, spikegen
    import snntorch.functional as SF
    HAS_SNNTORCH = True
    print(f'snntorch: {snn.__version__}')
except ImportError:
    HAS_SNNTORCH = False
    print('snntorch not found — using ReLU fallback')

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
        )
        self.pool = nn.MaxPool2d(2, 2)
    def forward(self, x):
        f = self.conv(x); return f, self.pool(f)

class SiameseEncoder(nn.Module):
    def __init__(self, in_channels=4, encoder_channels=None):
        super().__init__()
        if encoder_channels is None: encoder_channels = [64,128,256,512]
        chs = [in_channels] + encoder_channels
        self.blocks = nn.ModuleList([EncoderBlock(chs[i], chs[i+1]) for i in range(len(encoder_channels))])
        self.bottleneck = nn.Sequential(
            nn.Conv2d(encoder_channels[-1], encoder_channels[-1]*2, 3, padding=1, bias=False),
            nn.BatchNorm2d(encoder_channels[-1]*2), nn.ReLU(True),
            nn.Conv2d(encoder_channels[-1]*2, encoder_channels[-1]*2, 3, padding=1, bias=False),
            nn.BatchNorm2d(encoder_channels[-1]*2), nn.ReLU(True),
        )
    def encode_single(self, x):
        skips = []
        for b in self.blocks: f, x = b(x); skips.append(f)
        return self.bottleneck(x), skips
    def forward(self, a, b):
        ba,sa = self.encode_single(a); bb,sb = self.encode_single(b)
        return torch.abs(ba-bb), [torch.abs(x-y) for x,y in zip(sa,sb)]

class SNNDecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, beta=0.9):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv1 = nn.Sequential(nn.Conv2d(out_ch+skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch))
        self.conv2 = nn.Sequential(nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch))
        if HAS_SNNTORCH:
            sg = surrogate.fast_sigmoid(slope=25)
            self.lif1 = snn.Leaky(beta=beta, spike_grad=sg, init_hidden=False)
            self.lif2 = snn.Leaky(beta=beta, spike_grad=sg, init_hidden=False)
        else:
            self.lif1 = self.lif2 = None; self.relu = nn.ReLU(True)
    def forward(self, x, skip, m1=None, m2=None):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1); x = self.conv1(x)
        if HAS_SNNTORCH and self.lif1:
            if m1 is None: m1 = self.lif1.init_leaky()
            s1,m1 = self.lif1(x,m1); x = self.conv2(s1)
            if m2 is None: m2 = self.lif2.init_leaky()
            s2,m2 = self.lif2(x,m2); return s2,m1,m2
        else:
            x = self.relu(x); x = self.conv2(x); x = self.relu(x); return x,None,None

class SNNDecoder(nn.Module):
    def __init__(self, enc_chs=None, num_classes=2, beta=0.9, num_steps=10):
        super().__init__()
        if enc_chs is None: enc_chs = [64,128,256,512]
        self.num_steps = num_steps
        self.blocks = nn.ModuleList()
        in_ch = enc_chs[-1]*2
        for i in range(len(enc_chs)-1,-1,-1):
            self.blocks.append(SNNDecoderBlock(in_ch, enc_chs[i], enc_chs[i], beta)); in_ch = enc_chs[i]
        self.head = nn.Conv2d(enc_chs[0], num_classes, 1)
    def forward(self, spk_in, skips):
        T = spk_in.shape[0] if spk_in.dim()==5 else self.num_steps
        rev_skips = list(reversed(skips))
        spk_rec, mem_rec = [], []
        block_mems = [(None,None) for _ in self.blocks]
        for t in range(T):
            x = spk_in[t] if spk_in.dim()==5 else spk_in
            for i,(blk,sk) in enumerate(zip(self.blocks, rev_skips)):
                m1,m2 = block_mems[i]; x,m1,m2 = blk(x,sk,m1,m2); block_mems[i]=(m1,m2)
            out = self.head(x); spk_rec.append(out); mem_rec.append(out.clone())
        return spk_rec, mem_rec

class SiameseSNN(nn.Module):
    def __init__(self, in_channels=4, encoder_channels=None, num_classes=2, beta=0.9, num_steps=10):
        super().__init__()
        if encoder_channels is None: encoder_channels = [64,128,256,512]
        self.num_steps = num_steps
        self.encoder = SiameseEncoder(in_channels, encoder_channels)
        self.decoder = SNNDecoder(encoder_channels, num_classes, beta, num_steps)
    def forward(self, a, b):
        diff_bot, diff_skips = self.encoder(a, b)
        if HAS_SNNTORCH:
            spike_trains = spikegen.rate(torch.sigmoid(diff_bot), num_steps=self.num_steps)
        else:
            spike_trains = diff_bot.unsqueeze(0).repeat(self.num_steps,1,1,1,1)
        return self.decoder(spike_trains, diff_skips)
    def predict(self, a, b):
        self.eval()
        with torch.no_grad():
            spk_rec, _ = self.forward(a, b)
            logits = torch.stack(spk_rec, dim=0).mean(dim=0)
            return logits.argmax(dim=1)
    def get_change_prob(self, a, b):
        self.eval()
        with torch.no_grad():
            spk_rec, _ = self.forward(a, b)
            logits = torch.stack(spk_rec, dim=0).mean(dim=0)
            return torch.softmax(logits, dim=1)[:, 1]

print('✅ Model loaded')

In [ ]:
class ChangeDetectionLoss(nn.Module):
    def __init__(self, change_weight=15.0, dice_weight=0.3):
        super().__init__()
        self.dice_weight = dice_weight
        self.register_buffer('class_weights', torch.tensor([1.0, change_weight]))
    def forward(self, spk_recordings, targets):
        logits = torch.stack(spk_recordings, dim=0).mean(dim=0)
        ce = F.cross_entropy(logits, targets.long(), weight=self.class_weights.to(logits.device))
        probs = torch.softmax(logits, dim=1)[:, 1]
        tgt = targets.float()
        inter = (probs * tgt).sum()
        dice = 1.0 - (2*inter + 1) / (probs.sum() + tgt.sum() + 1)
        return ce + self.dice_weight * dice

print('✅ Loss loaded')

In [ ]:
# ============================================================
#  DATASET — FIXED: Only uses cities WITH labels
#  Split: 11 labeled cities for train, 3 for validation
# ============================================================
from torch.utils.data import Dataset
from pathlib import Path

try:
    import rasterio; HAS_RASTERIO = True
except: HAS_RASTERIO = False
try:
    from scipy.ndimage import zoom as scipy_zoom; HAS_SCIPY = True
except: HAS_SCIPY = False

# ALL 14 LABELED cities (the only ones with cm.png)
ALL_LABELED = ['aguasclaras','bercy','bordeaux','nantes','paris','rennes',
               'saclay_e','abudhabi','cupertino','pisa','beihai','hongkong','beirut','mumbai']

# Split: 11 train, 3 val (hold out diverse cities for validation)
TRAIN_CITIES = ['aguasclaras','bercy','bordeaux','nantes','paris','rennes',
                'saclay_e','cupertino','pisa','beihai','hongkong']
VAL_CITIES   = ['abudhabi','beirut','mumbai']  # diverse geographies for robust validation

def _find_band(d, bn):
    d = Path(d)
    p = d/f'{bn}.tif'
    if p.exists(): return p
    m = list(d.glob(f'*_{bn}.tif'))
    if m: return m[0]
    m = list(d.glob(f'*_{bn.upper()}.tif'))
    return m[0] if m else None

class OSCDDataset(Dataset):
    def __init__(self, root, split='train', patch_size=128, bands=None, augment=True):
        self.root = Path(root)
        self.split = split
        self.ps = patch_size
        self.bands = bands or ['B02','B03','B04','B08']
        self.augment = augment and split=='train'
        self._dims = {}

        # KEY FIX: Only use labeled cities, properly split
        if split == 'train':
            candidates = TRAIN_CITIES
        else:
            candidates = VAL_CITIES

        # Filter to cities that exist AND have labels
        self.cities = [c for c in candidates
                       if (self.root/c).exists() and (self.root/c/'cm'/'cm.png').exists()]

        if not self.cities:
            print(f'⚠️ No labeled cities found for {split}!')
            self.cities = []; self.patches = []; return

        self.patches = self._index()
        # Print change mask stats
        total_change = 0
        for c in self.cities:
            m = self._loadmask(c)
            total_change += m.mean()
        avg_change = total_change / len(self.cities) * 100
        print(f'{split}: {len(self.cities)} cities, {len(self.patches)} patches, avg change={avg_change:.1f}%')

    def _refdim(self, city):
        if city in self._dims: return self._dims[city]
        h,w = 512,512
        for b in ['B02','B03','B04','B08']:
            f = _find_band(self.root/city/'imgs_1', b)
            if f and HAS_RASTERIO:
                with rasterio.open(str(f)) as s: h,w = s.height, s.width
                break
        self._dims[city] = (h,w); return h,w

    def _index(self):
        patches = []
        for c in self.cities:
            h,w = self._refdim(c)
            stride = self.ps//2 if self.split=='train' else self.ps
            for r in range(0, h-self.ps+1, stride):
                for co in range(0, w-self.ps+1, stride):
                    patches.append({'city':c, 'row':r, 'col':co})
        return patches

    def _loadbands(self, city, td):
        d = self.root/city/td
        rh,rw = self._refdim(city)
        arrs = []
        for bn in self.bands:
            f = _find_band(d, bn)
            if f and HAS_RASTERIO:
                with rasterio.open(str(f)) as s: data = s.read(1).astype(np.float32)
                if data.shape != (rh,rw) and HAS_SCIPY:
                    data = scipy_zoom(data, (rh/data.shape[0], rw/data.shape[1]), order=1)[:rh,:rw]
                    if data.shape[0]<rh or data.shape[1]<rw:
                        data = np.pad(data, ((0,rh-data.shape[0]),(0,rw-data.shape[1])))
                arrs.append(data)
            else:
                arrs.append(np.zeros((rh,rw), dtype=np.float32))
        return np.stack(arrs, 0)

    def _loadmask(self, city):
        cm = self.root/city/'cm'/'cm.png'
        rh,rw = self._refdim(city)
        if cm.exists():
            from PIL import Image
            m = np.array(Image.open(str(cm)))
            if m.ndim == 3: m = m[:,:,0]
            m = (m > 128).astype(np.float32)
            if m.shape != (rh,rw):
                from PIL import Image as PI
                m = (np.array(PI.fromarray((m*255).astype(np.uint8)).resize((rw,rh), PI.NEAREST)) > 128).astype(np.float32)
        else:
            m = np.zeros((rh,rw), dtype=np.float32)
        return m

    def __len__(self): return len(self.patches)

    def __getitem__(self, idx):
        p = self.patches[idx]
        city, r, c = p['city'], p['row'], p['col']
        a = self._loadbands(city, 'imgs_1')
        b = self._loadbands(city, 'imgs_2')
        m = self._loadmask(city)
        a = a[:, r:r+self.ps, c:c+self.ps]
        b = b[:, r:r+self.ps, c:c+self.ps]
        m = m[r:r+self.ps, c:c+self.ps]
        _, ph, pw = a.shape
        if ph < self.ps or pw < self.ps:
            a = np.pad(a, ((0,0),(0,self.ps-ph),(0,self.ps-pw)))
            b = np.pad(b, ((0,0),(0,self.ps-ph),(0,self.ps-pw)))
            m = np.pad(m, ((0,self.ps-ph),(0,self.ps-pw)))
        for ch in range(a.shape[0]):
            for img in [a, b]:
                mn, mx = img[ch].min(), img[ch].max()
                if mx > mn: img[ch] = (img[ch]-mn)/(mx-mn)
        if self.augment:
            if np.random.random() > 0.5: a=np.flip(a,2).copy(); b=np.flip(b,2).copy(); m=np.flip(m,1).copy()
            if np.random.random() > 0.5: a=np.flip(a,1).copy(); b=np.flip(b,1).copy(); m=np.flip(m,0).copy()
            k = np.random.randint(0, 4)
            if k > 0: a=np.rot90(a,k,(1,2)).copy(); b=np.rot90(b,k,(1,2)).copy(); m=np.rot90(m,k,(0,1)).copy()
        return torch.from_numpy(a.copy()).float(), torch.from_numpy(b.copy()).float(), torch.from_numpy(m.copy()).long()

print('✅ Dataset ready (LABELED CITIES ONLY)')

In [ ]:
GPU_CONFIG = {
    'epochs': 60,
    'batch_size': 8,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'patch_size': 128,
    'num_steps': 10,
    'num_bands': 4,
    'encoder_channels': [32, 64, 128, 256],
    'gradient_clip': 1.0,
    'change_weight': 15.0,
}
USE_BANDS = ['B02','B03','B04','B08']
print('Config:', GPU_CONFIG)

In [ ]:
import time, os
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.amp import GradScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Dataset — both splits use LABELED cities only
train_ds = OSCDDataset(str(oscd_root), 'train', GPU_CONFIG['patch_size'], USE_BANDS, True)
val_ds   = OSCDDataset(str(oscd_root), 'val',   GPU_CONFIG['patch_size'], USE_BANDS, False)

train_loader = DataLoader(train_ds, batch_size=GPU_CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=GPU_CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
print(f'\nTrain: {len(train_ds)} patches ({len(train_loader)} batches)')
print(f'Val:   {len(val_ds)} patches ({len(val_loader)} batches)')

# Model
model = SiameseSNN(
    in_channels=GPU_CONFIG['num_bands'],
    encoder_channels=GPU_CONFIG['encoder_channels'],
    beta=0.9, num_steps=GPU_CONFIG['num_steps'],
).to(device)
print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')

loss_fn = ChangeDetectionLoss(change_weight=GPU_CONFIG['change_weight']).to(device)
optimizer = AdamW(model.parameters(), lr=GPU_CONFIG['lr'], weight_decay=GPU_CONFIG['weight_decay'])
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=1e-5)
scaler = GradScaler('cuda')

os.makedirs('/content/outputs/models', exist_ok=True)
best_f1 = 0.0
history = {'train_loss':[], 'val_loss':[], 'val_f1':[], 'val_iou':[]}

print('\n' + '='*70)
print(f'  🚀 Training: {GPU_CONFIG["epochs"]} epochs | LR={GPU_CONFIG["lr"]} | T={GPU_CONFIG["num_steps"]}')
print('='*70)

for epoch in range(1, GPU_CONFIG['epochs']+1):
    t0 = time.time()

    # Train
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        img_a, img_b, mask = batch[0].to(device), batch[1].to(device), batch[2].to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            spk_rec, _ = model(img_a, img_b)
            loss = loss_fn(spk_rec, mask)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GPU_CONFIG['gradient_clip'])
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    scheduler.step()
    avg_train = total_loss / len(train_loader)
    history['train_loss'].append(avg_train)

    # Validate
    model.eval()
    tp = fp = fn = tn = 0
    vl = 0.0
    with torch.no_grad():
        for batch in val_loader:
            ia, ib, mk = batch[0].to(device).float(), batch[1].to(device).float(), batch[2].to(device).long()
            with torch.amp.autocast('cuda'):
                sr, _ = model(ia, ib)
                vl += loss_fn(sr, mk).item()
            pred = model.predict(ia, ib)
            tp += ((pred==1)&(mk==1)).sum().item()
            fp += ((pred==1)&(mk==0)).sum().item()
            fn += ((pred==0)&(mk==1)).sum().item()

    prec = tp/(tp+fp+1e-8)
    rec  = tp/(tp+fn+1e-8)
    f1   = 2*prec*rec/(prec+rec+1e-8)
    iou  = tp/(tp+fp+fn+1e-8)
    avg_val = vl/max(len(val_loader),1)
    lr_now = optimizer.param_groups[0]['lr']

    history['val_loss'].append(avg_val)
    history['val_f1'].append(f1)
    history['val_iou'].append(iou)

    print(f'Ep {epoch:3d}/{GPU_CONFIG["epochs"]} | '
          f'TrL:{avg_train:.4f} VL:{avg_val:.4f} | '
          f'F1:{f1:.4f} IoU:{iou:.4f} P:{prec:.3f} R:{rec:.3f} | '
          f'LR:{lr_now:.2e} | {time.time()-t0:.0f}s')

    # Diagnostics
    if epoch % 5 == 1 or f1 > 0:
        with torch.no_grad():
            sa, sb = batch[0][:1].to(device), batch[1][:1].to(device)
            cp = model.get_change_prob(sa, sb)
            print(f'  📊 prob: min={cp.min():.4f} max={cp.max():.4f} mean={cp.mean():.4f} | pred_px={model.predict(sa,sb).sum().item()}')

    if f1 > best_f1:
        best_f1 = f1
        torch.save({'epoch':epoch, 'model_state_dict':model.state_dict(), 'best_f1':best_f1, 'config':GPU_CONFIG},
                   '/content/outputs/models/best_model.pt')
        print(f'  ★ New best! F1={best_f1:.4f}')

    if epoch % 10 == 0:
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),'best_f1':best_f1},
                   f'/content/outputs/models/checkpoint_{epoch}.pt')

print(f'\n{"="*70}\n✅ Done! Best F1: {best_f1:.4f}\n{"="*70}')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,3,figsize=(18,5))
axes[0].plot(history['train_loss'], label='Train', color='#2196F3')
axes[0].plot(history['val_loss'], label='Val', color='#F44336')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['val_f1'], color='#4CAF50', linewidth=2)
axes[1].axhline(y=best_f1, color='gold', linestyle='--', label=f'Best={best_f1:.4f}')
axes[1].set_title('F1 Score'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(history['val_iou'], color='#9C27B0', linewidth=2)
axes[2].set_title('IoU'); axes[2].grid(alpha=0.3)
plt.suptitle('Siamese-SNN Training (V3 — Labeled Validation)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('/content/outputs/training_curves.png', dpi=150); plt.show()

In [ ]:
from google.colab import files
files.download('/content/outputs/models/best_model.pt')
print('📥 Place in: outputs/models/best_model.pt')

In [ ]:
files.download('/content/outputs/training_curves.png')